# Computer Vision — Lab 2: The R-CNN Family, Step by Step
**Level:** Master's · **Duration:** two 3-hour sessions · **Environment:** Jupyter / Colab

**Goal:** discover *why* each architecture was introduced by inspecting intermediate tensors and visualizations.

**Teaching scope:** Steps 1–3 are **pedagogical approximations**, not complete trained R-CNN or Fast R-CNN detectors. Steps 4–5 use pretrained TorchVision detection models. This notebook performs **inference only**, not training.

**Recommended workflow:** Run cells from top to bottom, read the observations, change one parameter, and answer each checkpoint. Initial model weight downloads require Internet; CPU execution is possible but the detector stages may be slow.

## 0. Environment and reproducible test images
Install dependencies once (uncomment the following cell if needed). Match `torch` and `torchvision` versions using the official PyTorch installation selector; installation of OpenCV contrib provides Selective Search.

In [ ]:
# %pip install -U numpy matplotlib pillow scikit-image opencv-contrib-python
# %pip install torch torchvision  # use matching builds from https://pytorch.org/get-started/locally/

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn.functional as F
from skimage import data, transform
from torchvision import models
from torchvision.ops import roi_pool, roi_align
from torchvision.transforms.functional import to_tensor

np.random.seed(7)
torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "TorchVision:", __import__('torchvision').__version__)
print("OpenCV:", cv2.__version__, "Device:", device)
assert hasattr(cv2, "ximgproc") and hasattr(cv2.ximgproc, "segmentation"), (
    "Selective Search requires opencv-contrib-python."
)

In [ ]:
# Packaged sample pictures: no dataset account or API key needed.
# The composite offers multiple visual regions, but the pretrained detectors
# may not recognize every object in this synthetic composition.
left = transform.resize(data.astronaut(), (420, 420), preserve_range=True).astype(np.uint8)
right = transform.resize(data.chelsea(), (420, 420), preserve_range=True).astype(np.uint8)
image = np.concatenate([left, right], axis=1)
height, width = image.shape[:2]
plt.figure(figsize=(12, 5)); plt.imshow(image); plt.axis('off')
plt.title(f"Teaching image: {width} × {height}"); plt.show()

## 1. Region proposals: Selective Search
**Question:** How can a detector find promising object locations *before* recognizing objects?

Selective Search groups image regions according to visual similarity. It **does not** assign object categories. The number of proposed boxes can be much larger than the number of real objects.

In [ ]:
start = time.perf_counter()
ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
ss.setBaseImage(cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
ss.switchToSelectiveSearchFast()
raw_proposals = ss.process()  # each proposal is (x, y, width, height)
proposal_time = time.perf_counter() - start

# Remove tiny boxes and convert to (x1, y1, x2, y2).
boxes = []
for x, y, w, h in raw_proposals:
    if w >= 32 and h >= 32:
        boxes.append([int(x), int(y), int(x+w), int(y+h)])
boxes = np.asarray(boxes, dtype=np.int32)
print(f"Generated {len(raw_proposals)} proposals; kept {len(boxes)}; {proposal_time:.2f}s")
assert len(boxes) >= 10, "Unexpectedly few proposals: inspect the input image."

def show_boxes(rgb, coordinates, title, max_boxes=40):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(rgb)
    for x1, y1, x2, y2 in coordinates[:max_boxes]:
        ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                   fill=False, edgecolor='yellow', linewidth=0.7))
    ax.set_title(title); ax.axis('off'); plt.show()

show_boxes(image, boxes, "Selective Search candidate regions", 45)

**Checkpoint 1:** What do the rectangles represent? Does Selective Search know which ones correspond to cats or people? Change `max_boxes=45` to `max_boxes=10` and compare.

## 2. Original R-CNN: independent CNN feature extraction
**Original idea:** propose regions → crop/resize each region → run a CNN **separately** for each crop → classify and refine bounding boxes.

Below we demonstrate the expensive feature-extraction step using a pretrained MobileNet backbone (for runtime accessibility), **not the original R-CNN AlexNet/SVM training pipeline**.

In [ ]:
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
backbone_weights = MobileNet_V3_Small_Weights.DEFAULT
feature_extractor = mobilenet_v3_small(weights=backbone_weights).features.to(device).eval()

# Take a small number of region proposals to keep the lesson manageable.
N = 12
selected = boxes[:N]

def crop_to_tensor(rgb, rectangle, side=224):
    x1, y1, x2, y2 = map(int, rectangle)
    patch = rgb[y1:y2, x1:x2]
    patch = cv2.resize(patch, (side, side), interpolation=cv2.INTER_LINEAR)
    return to_tensor(patch)

patches = torch.stack([crop_to_tensor(image, b) for b in selected]).to(device)
fig, axs = plt.subplots(2, 6, figsize=(13, 5))
for ax, patch in zip(axs.flat, patches.cpu()):
    ax.imshow(patch.permute(1, 2, 0)); ax.axis('off')
plt.suptitle('Regions cropped and resized independently'); plt.show()

In [ ]:
def synchronize():
    if device.type == 'cuda':
        torch.cuda.synchronize()

individual_features = []
with torch.inference_mode():
    synchronize(); start = time.perf_counter()
    for patch in patches:
        feature_map = feature_extractor(patch.unsqueeze(0))
        individual_features.append(F.adaptive_avg_pool2d(feature_map, 1).flatten(1))
    synchronize(); independent_time = time.perf_counter()-start

individual_features = torch.cat(individual_features, dim=0)
print('Individual CNN calls:', len(selected))
print('Feature matrix:', tuple(individual_features.shape))
print(f'Independent crop processing: {independent_time:.3f} seconds')

**Checkpoint 2:** What happens to runtime when `N` increases? Why is it inefficient to process thousands of overlapping crops independently? **Note:** the example extracts features only; it does not implement R-CNN's SVM classifier, bounding-box regressors, or training procedure.

## 3. Fast R-CNN: share computation across proposed regions
Fast R-CNN computes one image feature map and extracts a fixed-size representation for each proposed region using **RoI Pooling**.

The demonstration below uses a single-scale MobileNet feature map instead of the exact original Fast R-CNN network. The point is to inspect the computational pattern.

In [ ]:
image_tensor = to_tensor(image).unsqueeze(0).to(device)
with torch.inference_mode():
    synchronize(); start = time.perf_counter()
    shared_map = feature_extractor(image_tensor)
    synchronize(); shared_cnn_time = time.perf_counter()-start

_, channels, feature_h, feature_w = shared_map.shape
print('Image tensor:', tuple(image_tensor.shape))
print('Shared feature map:', tuple(shared_map.shape))
print('Only one full-image CNN forward pass:', f'{shared_cnn_time:.3f}s')

plt.figure(figsize=(11, 4)); plt.imshow(shared_map[0].mean(0).cpu(), cmap='viridis')
plt.title('Mean activation over shared CNN feature channels'); plt.axis('off'); plt.show()

In [ ]:
# torchvision roi_pool accepts a list of xyxy tensors (one per image).
# For this backbone we use a single approximate spatial scale, suitable for
# the illustrative shared-feature example. Detector feature pyramids differ.
spatial_scale_x = feature_w / width
spatial_scale_y = feature_h / height
assert abs(spatial_scale_x - spatial_scale_y) < 0.01, 'Unexpected feature aspect scale.'

region_boxes = torch.as_tensor(selected, dtype=torch.float32, device=device)
with torch.inference_mode():
    synchronize(); start = time.perf_counter()
    roi_features = roi_pool(shared_map, [region_boxes], output_size=(7, 7),
                            spatial_scale=spatial_scale_x)
    synchronize(); roi_time = time.perf_counter()-start
print('RoI pooled features:', tuple(roi_features.shape))
print(f'Shared backbone + RoI Pool: {shared_cnn_time + roi_time:.3f}s')
print(f'Crop-by-crop features:   {independent_time:.3f}s')
print('Caution: runtime comparison is illustrative; receptive field, resize, '
      'feature content, and per-operation overhead differ.')

**Checkpoint 3:** Explain the shape `[N, channels, 7, 7]`. Which computation is performed only once? Why does Fast R-CNN still need an external proposal generator such as Selective Search?

## 4. Faster R-CNN: learn region proposals with an RPN
Faster R-CNN replaces the external Selective Search stage with a **Region Proposal Network** (RPN). An RPN predicts objectness and box adjustments for anchors on shared feature maps.

First we visualize a few anchor boxes using plain Python; then we run a pretrained Faster R-CNN detector and inspect intermediate RPN proposals.

In [ ]:
# Educational anchor visualization, not TorchVision's internal anchor generator.
cx, cy = width // 2, height // 2
anchor_sizes = [48, 96, 160]
aspect_ratios = [0.5, 1.0, 2.0]  # width/height
anchors = []
for area_side in anchor_sizes:
    for ratio in aspect_ratios:
        aw = area_side * np.sqrt(ratio)
        ah = area_side / np.sqrt(ratio)
        anchors.append([cx-aw/2, cy-ah/2, cx+aw/2, cy+ah/2])
show_boxes(image, np.asarray(anchors), 'Example anchors at one spatial location', 20)
print('Objectness: probability that an anchor describes some object.')
print('Box regression: predicted offsets refining the anchor.')

In [ ]:
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
)
from torchvision.utils import draw_bounding_boxes
weights_faster = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
faster = fasterrcnn_resnet50_fpn(weights=weights_faster).to(device).eval()
categories = weights_faster.meta['categories']

# Detector expects list of C×H×W float tensors in [0,1].
detector_input = [to_tensor(image).to(device)]
with torch.inference_mode():
    synchronize(); start = time.perf_counter()
    detections = faster(detector_input)[0]
    synchronize(); faster_time = time.perf_counter()-start

print('Prediction dictionary keys:', list(detections))
for key, val in detections.items():
    print(key, tuple(val.shape))
print(f'Inference time (first run): {faster_time:.2f}s')

In [ ]:
def visualize_detections(rgb, predictions, threshold=0.5):
    keep = predictions['scores'] >= threshold
    final_boxes = predictions['boxes'][keep].round().to(torch.int64).cpu()
    class_ids = predictions['labels'][keep].cpu().tolist()
    scores = predictions['scores'][keep].cpu().tolist()
    labels = [f'{categories[i]} {s:.2f}' for i, s in zip(class_ids, scores)]
    image_uint8 = torch.as_tensor(rgb.copy()).permute(2, 0, 1)
    if len(final_boxes):
        rendered = draw_bounding_boxes(image_uint8, final_boxes, labels,
                                       colors='red', width=3)
    else:
        rendered = image_uint8
    plt.figure(figsize=(14, 6)); plt.imshow(rendered.permute(1, 2, 0))
    plt.title(f'Faster R-CNN detections (confidence ≥ {threshold})')
    plt.axis('off'); plt.show()
    return int(keep.sum())

num_detections = visualize_detections(image, detections, threshold=0.5)
print('Retained detections:', num_detections)

In [ ]:
# Inspect the RPN proposals *before* final detection classification.
with torch.inference_mode():
    transformed_images, _ = faster.transform(detector_input, None)
    pyramid_features = faster.backbone(transformed_images.tensors)
    proposal_list, _ = faster.rpn(transformed_images, pyramid_features, None)

# Proposals are in the resized image coordinate system: map back to original.
resized_h, resized_w = transformed_images.image_sizes[0]
proposal_boxes = proposal_list[0][:35].detach().cpu().clone()
proposal_boxes[:, [0, 2]] *= width / resized_w
proposal_boxes[:, [1, 3]] *= height / resized_h
print('RPN proposals after filtering:', len(proposal_list[0]))
show_boxes(image, proposal_boxes.numpy(), 'RPN proposals (not final classifications)', 35)

In [ ]:
for confidence in [0.3, 0.5, 0.7, 0.9]:
    print(f'Confidence ≥ {confidence:.1f}:',
          int((detections['scores'] >= confidence).sum()))

**Checkpoint 4:** In the preceding experiment, which boxes are RPN proposals and which are final detections? What does an objectness score represent? Why are overlapping candidates filtered using non-maximum suppression?

## 5. Mask R-CNN: RoIAlign and a mask prediction branch
Mask R-CNN adds a per-instance mask branch and replaces the quantized RoI Pooling operation with **RoIAlign**, which samples features with bilinear interpolation.

We demonstrate the shape of an aligned feature tensor, then run pretrained Mask R-CNN and visualize masks.

In [ ]:
with torch.inference_mode():
    aligned = roi_align(shared_map, [region_boxes], output_size=(7, 7),
                        spatial_scale=spatial_scale_x, sampling_ratio=2, aligned=True)
print('RoI Pool tensor shape:', tuple(roi_features.shape))
print('RoIAlign tensor shape:', tuple(aligned.shape))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(roi_features[0].mean(0).cpu(), cmap='viridis')
axes[0].set_title('RoI Pooling (example)')
axes[1].imshow(aligned[0].mean(0).cpu(), cmap='viridis')
axes[1].set_title('RoIAlign (example)')
for ax in axes: ax.axis('off')
plt.show()

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
weights_mask = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
mask_model = maskrcnn_resnet50_fpn(weights=weights_mask).to(device).eval()
with torch.inference_mode():
    synchronize(); start = time.perf_counter()
    mask_predictions = mask_model(detector_input)[0]
    synchronize(); mask_time = time.perf_counter()-start

print('Mask R-CNN output keys:', list(mask_predictions))
for key, value in mask_predictions.items():
    print(key, tuple(value.shape))
print(f'Inference time (first run): {mask_time:.2f}s')

In [ ]:
def display_instance_masks(rgb, predictions, conf_threshold=0.5, mask_threshold=0.5):
    selected_indices = torch.where(predictions['scores'] >= conf_threshold)[0][:12]
    canvas = rgb.astype(np.float32).copy()
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(canvas.astype(np.uint8))
    colors = plt.cm.tab20(np.linspace(0, 1, max(len(selected_indices), 1)))
    for color, index in zip(colors, selected_indices.tolist()):
        probability = predictions['masks'][index, 0].detach().cpu().numpy()
        binary_mask = probability >= mask_threshold
        overlay = np.zeros((*binary_mask.shape, 4), dtype=float)
        overlay[binary_mask, :3] = color[:3]
        overlay[binary_mask, 3] = 0.4
        ax.imshow(overlay)
        x1, y1, x2, y2 = predictions['boxes'][index].detach().cpu().numpy()
        ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                   fill=False, edgecolor=color, linewidth=2))
        label_id = int(predictions['labels'][index])
        ax.text(x1, y1, f'{categories[label_id]}', color='white', fontsize=9,
                bbox=dict(facecolor=color, alpha=.8, edgecolor='none'))
    ax.axis('off')
    ax.set_title(f'Mask R-CNN: {len(selected_indices)} retained object instances')
    plt.show()

display_instance_masks(image, mask_predictions)

**Checkpoint 5:** What is the difference between the `boxes` and `masks` outputs? If two cats are present, how would two instance masks differ from a single semantic segmentation mask? Why is fine spatial alignment useful when predicting masks?

## 6. Final comparison and students' report
Fill the table below from the notebook and reading. The timing of our **pedagogical partial pipelines** is not a fair end-to-end detector benchmark. The detector models also differ in backbone, resolution, and output head; their first-run times include overhead.

| Question | R-CNN | Fast R-CNN | Faster R-CNN | Mask R-CNN |
|---|---|---|---|---|
| Proposal generation | Selective Search | Selective Search | Learned RPN | Learned RPN |
| Feature computation | Per cropped region | Shared image features | Shared image features | Shared image features |
| Region operation | Crop / warp | RoI Pool | RoI Pool (original) | RoIAlign |
| Predicts boxes | Yes | Yes | Yes | Yes |
| Predicts instance masks | No | No | No | Yes |

**Student tasks**
1. Save one visualization from each stage and describe its input/output.
2. Change the number of R-CNN proposals from 12 to 24; report feature-extraction runtimes.
3. Change the Faster R-CNN confidence threshold; report detection counts.
4. Explain why the external region generator was replaced by an RPN.
5. Explain why Mask R-CNN's mask branch needs accurately aligned region features.
6. State what is *not* implemented in the teaching approximations.

**Further reading:** Girshick et al. (2014) R-CNN; Girshick (2015) Fast R-CNN; Ren et al. (2015) Faster R-CNN; He et al. (2017) Mask R-CNN; TorchVision detection model documentation.